🌾 Andhra Pradesh Multi-Commodity Interactive Price Prediction Engine
### 18 Commodities | Interactive Dropdowns | 70/20/10 Split | Global XGBoost

**How to use:**
1. **Run all cells** (Runtime → Run all)
2. **Select a commodity** from the dropdown in Step 2
3. **Click "Fetch & Train"** to download data and train the model
4. **Select a market** from the dropdown in Step 3
5. **Click "Predict"** to see 3-day forecasts with dates

## Step 1: Setup & Dependencies

In [ ]:
import urllib.request
import urllib.parse
import json
import pandas as pd
import numpy as np
import time
import warnings
from datetime import datetime, timedelta
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
warnings.filterwarnings('ignore')

# API Configuration
API_KEY = "579b464db66ec23bdd000001a0a99e04a75a40666201931688acb738"
RESOURCE_ID = "35985678-0d79-46b4-9ed6-6f13308a1d24"
BASE_URL = f"https://api.data.gov.in/resource/{RESOURCE_ID}"
TARGET_STATE = "Andhra Pradesh"
TOP_N_MARKETS = 10
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.20, 0.10

VALID_COMMODITIES = [
    "Paddy(Common)", "Tomato", "Banana", "Maize", "Groundnut",
    "Dry Chillies", "Mango", "Gur(Jaggery)", "Lemon", "Cotton",
    "Turmeric", "Bengal Gram(Gram)(Whole)", "Chili Red",
    "Black Gram(Urd Beans)(Whole)", "Castor Seed", "Tamarind Fruit",
    "Lime", "Red gram/Arhar/Tur(whole)"
]

# Global state
model_state = {
    'models': {}, 'featured_df': None, 'FEATURES': [],
    'use_quantile': False, 'val_mae': 0, 'test_acc': 0,
    'test_mape': 0, 'commodity': None, 'markets': [],
    'trained': False
}

print("✅ Setup complete. Proceed to Step 2 to select a commodity.")

## Step 2: Select Commodity & Train Model

Choose your commodity from the dropdown and click **"Fetch & Train"**.

In [ ]:
def fetch_mandi_data(state, commodity, output_widget):
    all_records = []
    offset = 0
    limit = 1000
    total = None

    with output_widget:
        print(f"Fetching {commodity} data for {state}...")

    while True:
        params = {
            "api-key": API_KEY, "format": "json",
            "limit": limit, "offset": offset,
            "filters[state]": state, "filters[commodity]": commodity
        }
        url = f"{BASE_URL}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        try:
            with urllib.request.urlopen(req, timeout=15) as response:
                data = json.loads(response.read().decode('utf-8'))
                records = data.get("records", [])
                if total is None:
                    total = int(data.get("total", 0))
                if not records:
                    break
                valid = [r for r in records if r.get('State', '') == state]
                if len(valid) == 0:
                    break
                all_records.extend(valid)
                offset += limit
                if offset >= total:
                    break
                time.sleep(0.2)
        except Exception as e:
            time.sleep(1.0)
            try:
                with urllib.request.urlopen(req, timeout=15) as response:
                    data = json.loads(response.read().decode('utf-8'))
                    recs = data.get("records", [])
                    valid = [r for r in recs if r.get('State','') == state]
                    all_records.extend(valid)
                    offset += limit
            except:
                break

    df = pd.DataFrame(all_records)
    if 'State' in df.columns and not df.empty:
        df = df[df['State'] == state].reset_index(drop=True)
    return df

def create_features(df):
    df = df.sort_values(['Market', 'date']).copy()
    for lag in [1, 2, 3, 7]:
        df[f'lag_{lag}'] = df.groupby('Market')['weighted_avg_modal_price'].shift(lag)
    df['min_lag_1'] = df.groupby('Market')['min_price'].shift(1)
    df['max_lag_1'] = df.groupby('Market')['max_price'].shift(1)
    df['rolling_mean_3'] = df.groupby('Market')['lag_1'].transform(lambda x: x.rolling(3).mean())
    df['rolling_mean_7'] = df.groupby('Market')['lag_1'].transform(lambda x: x.rolling(7).mean())
    df['prev_spread'] = df['max_lag_1'] - df['min_lag_1']
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    return df.dropna()

def on_train_click(btn):
    import xgboost as xgb
    from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error

    with train_output:
        clear_output(wait=True)
        commodity = commodity_dropdown.value
        btn.disabled = True
        btn.description = "Training..."

        # ── Fetch ──
        raw_df = fetch_mandi_data(TARGET_STATE, commodity, train_output)
        if raw_df.empty:
            print(f"❌ No data found for {commodity}.")
            btn.disabled = False
            btn.description = "🚀 Fetch & Train"
            return

        print(f"✅ Fetched {len(raw_df)} records")

        # ── Clean ──
        df = raw_df.copy()
        df['date'] = pd.to_datetime(df['Arrival_Date'], format='%d/%m/%Y', errors='coerce')
        df['Market'] = df['Market'].astype(str).str.replace(' APMC', '').str.strip()
        df['weighted_avg_modal_price'] = pd.to_numeric(df['Modal_Price'], errors='coerce')
        df['min_price'] = pd.to_numeric(df['Min_Price'], errors='coerce')
        df['max_price'] = pd.to_numeric(df['Max_Price'], errors='coerce')
        if 'State' in df.columns:
            df = df[df['State'] == TARGET_STATE]
        df = df.dropna(subset=['date', 'weighted_avg_modal_price'])
        df = df[df['date'] >= '2018-01-01'].sort_values(['Market', 'date']).reset_index(drop=True)

        # Top N markets
        market_counts = df['Market'].value_counts()
        avail_n = min(TOP_N_MARKETS, len(market_counts))
        top_mkts = market_counts.head(avail_n).index.tolist()
        df = df[df['Market'].isin(top_mkts)].reset_index(drop=True)

        print(f"✅ {commodity}: {len(df)} rows, {avail_n} markets (2018+)")
        print()
        for mkt in top_mkts:
            cnt = market_counts[mkt]
            dr = df[df['Market'] == mkt]['date']
            print(f"  {mkt:35s} {cnt:4d} rows  ({dr.min().strftime('%Y-%m-%d')} to {dr.max().strftime('%Y-%m-%d')})")

        # ── Features ──
        featured = create_features(df)
        featured = pd.get_dummies(featured, columns=['Market'], prefix='mkt', drop_first=False)

        exclude = ['date', 'weighted_avg_modal_price', 'min_price', 'max_price',
                    'District', 'State', 'Commodity', 'Variety', 'Grade',
                    'Arrival_Date', 'Market', 'Modal_Price', 'Min_Price', 'Max_Price',
                    'Commodity_Code']
        FEATURES = [c for c in featured.columns if c not in exclude and featured[c].dtype != 'object']

        # ── 70/20/10 Split ──
        sdates = featured['date'].sort_values()
        tc = sdates.quantile(TRAIN_RATIO)
        vc = sdates.quantile(TRAIN_RATIO + VAL_RATIO)
        train_d = featured[featured['date'] <= tc]
        val_d = featured[(featured['date'] > tc) & (featured['date'] <= vc)]
        test_d = featured[featured['date'] > vc]
        X_tr, y_tr = train_d[FEATURES], train_d['weighted_avg_modal_price']
        X_va, y_va = val_d[FEATURES], val_d['weighted_avg_modal_price']
        X_te, y_te = test_d[FEATURES], test_d['weighted_avg_modal_price']

        print(f"\nSplit: Train={len(X_tr)} | Val={len(X_va)} | Test={len(X_te)}")

        # ── Train ──
        models = {}
        val_mae_val = 0
        uq = False
        try:
            print("Training Quantile XGBoost...")
            for q, alpha in [('p50', 0.5), ('p10', 0.1), ('p90', 0.9)]:
                models[q] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=alpha,
                    max_depth=5, learning_rate=0.05, n_estimators=500,
                    random_state=42, early_stopping_rounds=20)
                models[q].fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            uq = True
            print(f"✅ Quantile models trained (best iter: {models['p50'].best_iteration})")
        except:
            print("[FALLBACK] Using squared error.")
            models['p50'] = xgb.XGBRegressor(objective='reg:squarederror',
                max_depth=5, learning_rate=0.05, n_estimators=500,
                random_state=42, early_stopping_rounds=20)
            models['p50'].fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            val_mae_val = np.mean(np.abs(y_va - models['p50'].predict(X_va)))

        # ── Evaluate ──
        def ev(X, y):
            p = models['p50'].predict(X)
            mape = mean_absolute_percentage_error(y, p) * 100
            mae = mean_absolute_error(y, p)
            return round(max(0, 100-mape), 2), round(mape, 2), round(mae, 2)

        tr_a, tr_m, tr_mae = ev(X_tr, y_tr)
        va_a, va_m, va_mae = ev(X_va, y_va)
        te_a, te_m, te_mae = ev(X_te, y_te)
        bl_mape = mean_absolute_percentage_error(y_te, X_te['lag_1']) * 100

        print()
        print("=" * 70)
        print(f"  MODEL PERFORMANCE: {commodity}")
        print("=" * 70)
        print(f"  {'Split':<20} {'Rows':>6} {'Accuracy':>10} {'MAPE':>8} {'MAE (Rs)':>10}")
        print(f"  {'-'*56}")
        print(f"  {'Train (70%)':<20} {len(X_tr):>6} {tr_a:>9.2f}% {tr_m:>7.2f}% {tr_mae:>10.2f}")
        print(f"  {'Validation (20%)':<20} {len(X_va):>6} {va_a:>9.2f}% {va_m:>7.2f}% {va_mae:>10.2f}")
        print(f"  {'⭐ TEST (10%)':<20} {len(X_te):>6} {te_a:>9.2f}% {te_m:>7.2f}% {te_mae:>10.2f}")
        print(f"  {'-'*56}")
        print(f"  Baseline (Lag-1) Test MAPE: {bl_mape:.2f}%")
        gap = te_m - tr_m
        print(f"  Generalization Gap: {gap:+.2f}%")

        if te_a >= 90: grade = "🟢 EXCELLENT"
        elif te_a >= 80: grade = "🟡 GOOD"
        elif te_a >= 70: grade = "🟠 FAIR"
        else: grade = "🔴 POOR"
        print(f"\n  ⭐ FINAL TEST ACCURACY: {te_a:.2f}% {grade}")

        # Save state
        model_state['models'] = models
        model_state['featured_df'] = featured
        model_state['FEATURES'] = FEATURES
        model_state['use_quantile'] = uq
        model_state['val_mae'] = val_mae_val
        model_state['test_acc'] = te_a
        model_state['test_mape'] = te_m
        model_state['commodity'] = commodity
        model_state['markets'] = sorted([c.replace('mkt_', '') for c in FEATURES if c.startswith('mkt_')])
        model_state['trained'] = True

        # Update market dropdown in Step 3
        market_dropdown.options = model_state['markets']
        if model_state['markets']:
            market_dropdown.value = model_state['markets'][0]

        btn.disabled = False
        btn.description = "🚀 Fetch & Train"
        print(f"\n✅ Ready! Go to Step 3 to select a market and predict.")

# ── Build UI ──
commodity_dropdown = widgets.Dropdown(
    options=VALID_COMMODITIES,
    value='Paddy(Common)',
    description='Commodity:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

train_btn = widgets.Button(
    description='🚀 Fetch & Train',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px'),
    style={'font_weight': 'bold'}
)
train_btn.on_click(on_train_click)

train_output = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<h3>🔍 Select Commodity</h3>'),
    commodity_dropdown,
    train_btn,
    train_output
]))

## Step 3: Select Market & Predict

After training completes above, select a market and click **"Predict 3-Day Forecast"**.

In [ ]:
def predict_3day(market_name):
    ms = model_state
    if not ms['trained']:
        return None, "Model not trained. Run Step 2 first."

    featured = ms['featured_df']
    FEATURES = ms['FEATURES']
    models = ms['models']

    mkt_col = f'mkt_{market_name}'
    if mkt_col not in featured.columns:
        return None, f"Market '{market_name}' not found."

    mkt_hist = featured[featured[mkt_col] == 1].sort_values('date')
    if mkt_hist.empty:
        return None, f"No data for {market_name}."

    latest_row = mkt_hist.iloc[-1].copy()
    last_data_date = mkt_hist['date'].max()
    last_price = float(latest_row['weighted_avg_modal_price'])
    forecast_base = max(pd.Timestamp(datetime.now().date()), last_data_date)

    predictions = []
    for day in range(1, 4):
        pred_date = forecast_base + timedelta(days=day)
        X_live = pd.DataFrame([latest_row])[FEATURES]
        p50 = float(models['p50'].predict(X_live)[0])

        if ms['use_quantile']:
            p10 = float(models['p10'].predict(X_live)[0])
            p90 = float(models['p90'].predict(X_live)[0])
        else:
            p10 = p50 - (1.5 * ms['val_mae'])
            p90 = p50 + (1.5 * ms['val_mae'])
        p10 = min(p10, p50)
        p90 = max(p90, p50)

        change = p50 - last_price
        cpct = (change / last_price) * 100 if last_price > 0 else 0

        if cpct > 2: trend = '📈 BULLISH'
        elif cpct < -2: trend = '📉 BEARISH'
        else: trend = '➡️ STABLE'

        predictions.append({
            'Forecast Date': pred_date.strftime('%Y-%m-%d (%A)'),
            'Expected (Rs/Q)': round(p50, 2),
            'Worst Case': round(p10, 2),
            'Best Case': round(p90, 2),
            'Change': f'{change:+.2f} ({cpct:+.1f}%)',
            'Trend': trend,
        })

        latest_row['lag_7'] = latest_row.get('lag_6', latest_row['lag_3'])
        latest_row['lag_3'] = latest_row['lag_2']
        latest_row['lag_2'] = latest_row['lag_1']
        latest_row['lag_1'] = p50
        latest_row['rolling_mean_3'] = (latest_row['lag_1'] + latest_row['lag_2'] + latest_row['lag_3']) / 3.0
        latest_row['rolling_mean_7'] = (latest_row['lag_1'] * 3 + latest_row['rolling_mean_3'] * 4) / 7.0

    meta = {
        'market': market_name, 'last_data_date': last_data_date.strftime('%Y-%m-%d'),
        'last_price': last_price, 'forecast_base': forecast_base.strftime('%Y-%m-%d'),
    }
    return pd.DataFrame(predictions), meta

def on_predict_click(btn):
    with predict_output:
        clear_output(wait=True)
        if not model_state['trained']:
            print("❌ Please train the model first (Step 2).")
            return

        market = market_dropdown.value
        result, meta = predict_3day(market)
        if result is None:
            print(f"❌ {meta}")
            return

        staleness = (pd.Timestamp(datetime.now().date()) - pd.Timestamp(meta['last_data_date'])).days
        stale = f"  ⚠️ Data is {staleness} days old" if staleness > 7 else ""

        print("=" * 85)
        print(f"  🌾 {model_state['commodity']} — {market} — 3-DAY FORECAST")
        print(f"  Test Accuracy: {model_state['test_acc']:.2f}%  |  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print("=" * 85)
        print(f"  Last Data: {meta['last_data_date']}  |  Last Price: Rs. {meta['last_price']:.2f}{stale}")
        print()
        print(result.to_string(index=False))
        print("=" * 85)

def on_predict_all_click(btn):
    with predict_output:
        clear_output(wait=True)
        if not model_state['trained']:
            print("❌ Please train the model first (Step 2).")
            return

        today = datetime.now()
        print("=" * 95)
        print(f"  🌾 {model_state['commodity']} — ALL {len(model_state['markets'])} AP MARKETS — 3-DAY FORECAST")
        print(f"  Test Accuracy: {model_state['test_acc']:.2f}%  |  Generated: {today.strftime('%Y-%m-%d %H:%M')}")
        print("=" * 95)

        summary = []
        for mkt in model_state['markets']:
            result, meta = predict_3day(mkt)
            if result is None:
                continue
            staleness = (pd.Timestamp(today.date()) - pd.Timestamp(meta['last_data_date'])).days
            sw = f" ⚠️({staleness}d)" if staleness > 7 else ""

            print(f"\n🏪 {mkt}{sw}")
            print(f"   Last: {meta['last_data_date']}  |  Price: Rs. {meta['last_price']:.2f}")
            print(result.to_string(index=False))
            print("-" * 95)

            for _, row in result.iterrows():
                summary.append({
                    'Market': mkt, 'Date': row['Forecast Date'],
                    'Expected': row['Expected (Rs/Q)'],
                    'Range': f"[{row['Worst Case']} - {row['Best Case']}]",
                    'Trend': row['Trend'],
                })

        print()
        print("=" * 95)
        print("  SUMMARY TABLE")
        print("=" * 95)
        print(pd.DataFrame(summary).to_string(index=False))

# ── Build UI ──
market_dropdown = widgets.Dropdown(
    options=['(Train model first)'],
    description='Market:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

predict_btn = widgets.Button(
    description='📈 Predict Selected Market',
    button_style='primary',
    layout=widgets.Layout(width='250px', height='40px')
)
predict_btn.on_click(on_predict_click)

predict_all_btn = widgets.Button(
    description='📊 Predict All Markets',
    button_style='info',
    layout=widgets.Layout(width='250px', height='40px')
)
predict_all_btn.on_click(on_predict_all_click)

predict_output = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<h3>🎯 Select Market & Predict</h3>'),
    market_dropdown,
    widgets.HBox([predict_btn, predict_all_btn]),
    predict_output
]))

## Step 4: Export JSON (Optional)

Run this cell to get structured JSON output for all markets.

In [ ]:
if not model_state['trained']:
    print("❌ Train the model first (Step 2).")
else:
    import json as json_lib
    api_out = {
        'commodity': model_state['commodity'],
        'state': TARGET_STATE,
        'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model': 'Global XGBoost (Quantile)' if model_state['use_quantile'] else 'Global XGBoost (Squared Error)',
        'split': '70/20/10', 'test_accuracy': model_state['test_acc'],
        'test_mape': model_state['test_mape'], 'markets': {}
    }
    for mkt in model_state['markets']:
        result, meta = predict_3day(mkt)
        if result is not None:
            api_out['markets'][mkt] = {
                'last_data_date': meta['last_data_date'],
                'last_price': meta['last_price'],
                'forecasts': result.to_dict(orient='records')
            }
    print(json_lib.dumps(api_out, indent=2, ensure_ascii=False))